In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC # Using SVC for classification instead of OneClassSVM
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

# --- Step 0: Load Data and Prepare for Evaluation ---

# Load the dataset
print("Loading data...")
df = pd.read_csv('Dataset_Tweets_Facts_01-Sheet1.csv')

# Drop rows where 'Tweet' or 'Label' is missing to avoid errors
df.dropna(subset=['Tweet', 'Label'], inplace=True)

# Check the distribution of labels
label_counts = df['Label'].value_counts()
print("Label distribution before handling rare classes:")
print(label_counts)

# Identify and remove classes with only one member
rare_classes = label_counts[label_counts < 2].index
df = df[~df['Label'].isin(rare_classes)]

print("\nLabel distribution after removing rare classes:")
print(df['Label'].value_counts())

# Define features (X) and target (y)
X = df['Tweet']
y = df['Label']

# Split the data into training and testing sets (80% train, 20% test)
# This is crucial for properly evaluating the models on unseen data.
# Stratify is used to maintain the proportion of each class in both splits.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nData split into {len(X_train)} training samples and {len(X_test)} testing samples.")

# --- Model 1: TF-IDF + Logistic Regression ---

print("\n--- Training Model 1: TF-IDF + Logistic Regression ---")

# Create a TF-IDF Vectorizer
# This converts text into numerical vectors based on word frequency.
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limit features to top 5000 words

# Fit the vectorizer on the training data and transform it
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
# Only transform the test data using the already-fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Create and train the Logistic Regression model
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Evaluate the model
y_pred_lr = lr_model.predict(X_test_tfidf)
print("Evaluation for TF-IDF + Logistic Regression:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

# Save the model and the vectorizer
joblib.dump(lr_model, 'tfidf_lr_classifier.joblib')
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.joblib')
print("✅ TF-IDF + Logistic Regression model saved.")


# --- Model 2: TF-IDF + Support Vector Machine (SVM) ---

print("\n--- Training Model 2: TF-IDF + SVM (SVC) ---")
# Note: We use SVC (Support Vector Classifier) for this task because it's designed for
# classifying labeled data. One-Class SVM is for unsupervised anomaly detection.

# We can reuse the TF-IDF vectors from the previous step
# Create and train the SVM model
svm_model = SVC(class_weight='balanced', kernel='linear', probability=True) # Using linear kernel as it's often good for text
svm_model.fit(X_train_tfidf, y_train)

# Evaluate the model
y_pred_svm = svm_model.predict(X_test_tfidf)
print("Evaluation for TF-IDF + SVM:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(classification_report(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))

# Save the model and the vectorizer (vectorizer is the same as above)
joblib.dump(svm_model, 'tfidf_svm_classifier.joblib')
print("✅ TF-IDF + SVM model saved.")


# --- Model 3: Sentence Transformers + Logistic Regression ---

print("\n--- Training Model 3: Sentence Transformers + Logistic Regression ---")

# Load a pre-trained Sentence Transformer model
print("Loading Sentence Transformer model...")
st_encoder = SentenceTransformer('all-mpnet-base-v2')

# Create embeddings for the tweets (this can take a few moments)
print("Creating embeddings for training data...")
X_train_st = st_encoder.encode(X_train.tolist(), show_progress_bar=True)
print("Creating embeddings for testing data...")
X_test_st = st_encoder.encode(X_test.tolist(), show_progress_bar=True)

# Create and train the Logistic Regression model
st_lr_model = LogisticRegression(class_weight='balanced', max_iter=1000)
st_lr_model.fit(X_train_st, y_train)

# Evaluate the model
y_pred_st_lr = st_lr_model.predict(X_test_st)
print("Evaluation for Sentence Transformers + Logistic Regression:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_st_lr):.4f}")
print(classification_report(y_test, y_pred_st_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_st_lr))

# Save the model and the encoder
joblib.dump(st_lr_model, 'st_lr_classifier.joblib')
st_encoder.save('st_encoder_model')
print("✅ Sentence Transformer + Logistic Regression model and encoder saved.")

Loading data...
Label distribution before handling rare classes:
Label
TRUE           635
FALSE          632
Half-true      173
Half-True       95
Mostly-True     92
Pants-fire      58
Pants-Fire      50
mostly-true     29
HALF-TRUE       24
half-true       20
Barely-True     19
pants-fire       4
barely-true      4
Speculative      2
speculative      1
irrelevant       1
PANTS-FIRE       1
Name: count, dtype: int64

Label distribution after removing rare classes:
Label
TRUE           635
FALSE          632
Half-true      173
Half-True       95
Mostly-True     92
Pants-fire      58
Pants-Fire      50
mostly-true     29
HALF-TRUE       24
half-true       20
Barely-True     19
barely-true      4
pants-fire       4
Speculative      2
Name: count, dtype: int64

Data split into 1469 training samples and 368 testing samples.

--- Training Model 1: TF-IDF + Logistic Regression ---
Evaluation for TF-IDF + Logistic Regression:
Accuracy: 0.5326
              precision    recall  f1-score   suppo

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Evaluation for TF-IDF + SVM:
Accuracy: 0.5978
              precision    recall  f1-score   support

 Barely-True       1.00      0.25      0.40         4
       FALSE       0.62      0.76      0.68       126
   HALF-TRUE       0.00      0.00      0.00         5
   Half-True       0.35      0.58      0.44        19
   Half-true       0.52      0.46      0.48        35
 Mostly-True       0.54      0.78      0.64        18
  Pants-Fire       0.33      0.40      0.36        10
  Pants-fire       0.80      0.33      0.47        12
        TRUE       0.84      0.55      0.67       127
 barely-true       0.00      0.00      0.00         1
   half-true       0.67      0.50      0.57         4
 mostly-true       0.10      0.33      0.15         6
  pants-fire       0.00      0.00      0.00         1

    accuracy                           0.60       368
   macro avg       0.44      0.38      0.37       368
weighted avg       0.65      0.60      0.60       368

Confusion Matrix:
 [[ 1  0  0  1 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/huggingfa

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

Creating embeddings for training data...


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Creating embeddings for testing data...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Evaluation for Sentence Transformers + Logistic Regression:
Accuracy: 0.4103
              precision    recall  f1-score   support

 Barely-True       0.25      0.50      0.33         4
       FALSE       0.70      0.28      0.40       126
   HALF-TRUE       0.11      0.60      0.19         5
   Half-True       0.26      0.37      0.30        19
   Half-true       0.34      0.49      0.40        35
 Mostly-True       0.44      0.83      0.58        18
  Pants-Fire       0.21      0.60      0.31        10
  Pants-fire       0.20      0.58      0.30        12
        TRUE       0.85      0.41      0.55       127
 barely-true       0.00      0.00      0.00         1
   half-true       0.20      0.50      0.29         4
 mostly-true       0.15      0.83      0.25         6
  pants-fire       0.00      0.00      0.00         1

    accuracy                           0.41       368
   macro avg       0.29      0.46      0.30       368
weighted avg       0.62      0.41      0.44       368

Co